# CMGAN Training Notebook

Notebook version of the training logic from train.py.

This version also includes Learning without Forgetting (LwF) with configurable $\lambda_{LwF}$ and `n_reset` for the snapshot interval.

In [2]:
%pip install torchinfo natsort einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [einops]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import csv
import logging
import os
from collections import OrderedDict

import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
from torchinfo import summary

from data import dataloader
from models import discriminator
from models.generator import TSCNet
from utils import power_compress, power_uncompress

logging.basicConfig(level=logging.INFO)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

ModuleNotFoundError: No module named 'torch'

In [ ]:
epochs = 120
batch_size = 4
log_interval = 500
decay_epoch = 30
init_lr = 5e-4
cut_len = 16000 * 2
data_dir = "/home/lugra002/speechbrain/data/cmgan_voicebank"
save_model_dir = "./saved_model"
metrics_csv_name = "train_metrics.csv"
loss_weights = [0.1, 0.9, 0.2, 0.05]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = 2

# LwF settings (Li & Hoiem, 2017)
lwf_enabled = True
lambda_lwf = 1.0
lambda_lwf_decay = 1.0
# Optional linear schedule override (takes precedence over lambda_lwf/lambda_lwf_decay).
lambda_lwf_start = None
lambda_lwf_end = None
n_reset = 10
kd_temperature = 2.0

def load_data_notebook(ds_dir, batch_size, num_workers, cut_len):
    torchaudio.set_audio_backend("sox_io")
    train_dir = os.path.join(ds_dir, "train")
    test_dir = os.path.join(ds_dir, "test")

    train_ds = dataloader.DemandDataset(train_dir, cut_len)
    test_ds = dataloader.DemandDataset(test_dir, cut_len)

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
    )

    return train_loader, test_loader

NameError: name 'torch' is not defined

In [ ]:
import os

# LwF ablations: vary the weight and the snapshot interval.
ABLATION_PRESETS = {
    "lwf_l0p5_n50": {
        "lwf_enabled": True,
        "lambda_lwf": 0.5,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 50,
    },
    "lwf_l0p5_n10": {
        "lwf_enabled": True,
        "lambda_lwf": 0.5,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 10,
    },
    "lwf_l0p7_n10": {
        "lwf_enabled": True,
        "lambda_lwf": 0.7,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 10,
    },
    "lwf_l0p7_n50": {
        "lwf_enabled": True,
        "lambda_lwf": 0.7,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 50,
    },
}

def apply_ablation_preset(name, results_root="./saved_model_ablations"):
    global lwf_enabled, lambda_lwf, lambda_lwf_decay, lambda_lwf_start, lambda_lwf_end, n_reset
    global save_model_dir, metrics_csv_name

    if name not in ABLATION_PRESETS:
        raise ValueError(f"Unknown preset '{name}'. Available: {list(ABLATION_PRESETS.keys())}")

    cfg = ABLATION_PRESETS[name]
    lwf_enabled = cfg["lwf_enabled"]
    lambda_lwf = cfg["lambda_lwf"]
    lambda_lwf_decay = cfg["lambda_lwf_decay"]
    lambda_lwf_start = cfg["lambda_lwf_start"]
    lambda_lwf_end = cfg["lambda_lwf_end"]
    n_reset = cfg["n_reset"]

    save_model_dir = os.path.join(results_root, name)
    metrics_csv_name = "train_metrics.csv"

    print("Applied ablation:")
    print(f"  preset          : {name}")
    print(f"  lwf_enabled     : {lwf_enabled}")
    print(f"  lambda_lwf      : {lambda_lwf}")
    print(f"  lambda_lwf_decay: {lambda_lwf_decay}")
    print(f"  lambda_start    : {lambda_lwf_start}")
    print(f"  lambda_end      : {lambda_lwf_end}")
    print(f"  n_reset         : {n_reset}")
    print(f"  save_model_dir  : {save_model_dir}")

print("Available presets:")
for preset_name in ABLATION_PRESETS:
    print(" -", preset_name)

Available presets:
 - baseline_no_lwf
 - lwf_l0p5_n10
 - lwf_l1p0_n10
 - lwf_l1p0_n50


## Ablation Presets

- `lwf_l0p5_n10`: LwF with lambda = 0.5 and `n_reset = 10`
- `lwf_l0p7_n10`: LwF with lambda = 0.7 and `n_reset = 10`
- `lwf_l0p5_n50`: LwF with lambda = 0.5 and `n_reset = 50`
- `lwf_l0p7_n50`: LwF with lambda = 0.7 and `n_reset = 50`

In [ ]:
    def _init_epoch_tracker(self):
        return {
            "steps": 0,
            "gen_loss": 0.0,
            "disc_loss": 0.0,
            "loss_ri": 0.0,
            "loss_mag": 0.0,
            "time_loss": 0.0,
            "gen_loss_gan": 0.0,
            "lwf_gen_loss": 0.0,
            "lwf_disc_loss": 0.0,
            "lwf_lambda_sum": 0.0,
            "pesq_sum": 0.0,
            "pesq_count": 0,
        }

    def _update_epoch_tracker(self, tracker, step_stats):
        tracker["steps"] += 1
        tracker["gen_loss"] += step_stats["gen_loss"]
        tracker["disc_loss"] += step_stats["disc_loss"]
        tracker["loss_ri"] += step_stats["loss_ri"]
        tracker["loss_mag"] += step_stats["loss_mag"]
        tracker["time_loss"] += step_stats["time_loss"]
        tracker["gen_loss_gan"] += step_stats["gen_loss_gan"]
        tracker["lwf_gen_loss"] += step_stats["lwf_gen_loss"]
        tracker["lwf_disc_loss"] += step_stats["lwf_disc_loss"]
        tracker["lwf_lambda_sum"] += step_stats.get("lwf_lambda", 0.0)

        if step_stats["pesq_mean"] is not None:
            tracker["pesq_sum"] += step_stats["pesq_mean"]
            tracker["pesq_count"] += 1

    def _finalize_epoch_tracker(self, tracker):
        steps = max(tracker["steps"], 1)
        stats = {
            "gen_loss": tracker["gen_loss"] / steps,
            "disc_loss": tracker["disc_loss"] / steps,
            "loss_ri": tracker["loss_ri"] / steps,
            "loss_mag": tracker["loss_mag"] / steps,
            "time_loss": tracker["time_loss"] / steps,
            "gen_loss_gan": tracker["gen_loss_gan"] / steps,
            "lwf_gen_loss": tracker["lwf_gen_loss"] / steps,
            "lwf_disc_loss": tracker["lwf_disc_loss"] / steps,
            "lwf_lambda": tracker["lwf_lambda_sum"] / steps,
            "pesq_mean": None,
        }
        if tracker["pesq_count"] > 0:
            stats["pesq_mean"] = tracker["pesq_sum"] / tracker["pesq_count"]
        return stats

    def train(self):
        scheduler_G = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=decay_epoch, gamma=0.5
        )
        scheduler_D = torch.optim.lr_scheduler.StepLR(
            self.optimizer_disc, step_size=decay_epoch, gamma=0.5
        )

        csv_ready = False
        for epoch in range(epochs):
            self.model.train()
            self.discriminator.train()
            self.maybe_refresh_lwf_snapshots(epoch)

            train_tracker = self._init_epoch_tracker()

            for idx, batch in enumerate(self.train_ds):
                step = idx + 1
                step_stats = self.train_step(batch, epoch)
                self._update_epoch_tracker(train_tracker, step_stats)

                if (step % log_interval) == 0:
                    logging.info(
                        "Epoch %d, Step %d, gen_loss=%.4f, disc_loss=%.4f, pesq=%.4f, lwf_lambda=%.4f",
                        epoch,
                        step,
                        step_stats["gen_loss"],
                        step_stats["disc_loss"],
                        step_stats["pesq_mean"],
                        step_stats["lwf_lambda"],
                    )

            train_stats = self._finalize_epoch_tracker(train_tracker)

            # Evaluation at end of epoch
            self.model.eval()
            self.discriminator.eval()

            test_tracker = self._init_epoch_tracker()
            for batch in self.test_ds:
                step_stats = self.test_step(batch)
                self._update_epoch_tracker(test_tracker, step_stats)

            test_stats = self._finalize_epoch_tracker(test_tracker)

            # Log and save
            logging.info(
                "Epoch %d completed: train_loss=%.4f, test_loss=%.4f, test_pesq=%.4f, lwf_lambda=%.4f",
                epoch,
                train_stats["gen_loss"],
                test_stats["gen_loss"],
                test_stats["pesq_mean"] or 0.0,
                train_stats["lwf_lambda"],
            )

            save_path = os.path.join(save_model_dir, f"model_epoch_{epoch:03d}.pt")
            os.makedirs(save_model_dir, exist_ok=True)
            torch.save(self.model.state_dict(), save_path)

            # CSV logging
            if not csv_ready:
                os.makedirs(save_model_dir, exist_ok=True)
                csv_ready = True

            with open(self.metrics_csv_path, "a", newline="") as f:
                writer = csv.DictWriter(
                    f,
                    fieldnames=[
                        "epoch",
                        "train_gen_loss",
                        "train_disc_loss",
                        "train_pesq",
                        "train_lwf_lambda",
                        "test_gen_loss",
                        "test_disc_loss",
                        "test_pesq",
                    ],
                )
                if epoch == 0:
                    writer.writeheader()
                writer.writerow(
                    {
                        "epoch": epoch,
                        "train_gen_loss": train_stats["gen_loss"],
                        "train_disc_loss": train_stats["disc_loss"],
                        "train_pesq": train_stats["pesq_mean"] or 0.0,
                        "train_lwf_lambda": train_stats["lwf_lambda"],
                        "test_gen_loss": test_stats["gen_loss"],
                        "test_disc_loss": test_stats["disc_loss"],
                        "test_pesq": test_stats["pesq_mean"] or 0.0,
                    }
                )

            scheduler_G.step()
            scheduler_D.step()

NameError: name 'torch' is not defined

In [ ]:
train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
trainer = Trainer(train_ds, test_ds)
trainer

In [ ]:
# Run the LwF ablations one after another.
PILOT_ABLATION_ORDER = [
    "lwf_l0p5_n10",
    "lwf_l0p7_n10",
    "lwf_l0p5_n50",
    "lwf_l0p7_n50",
]

# Reduce the epoch count here for quick tests.
# epochs = 20

for preset_name in PILOT_ABLATION_ORDER:
    apply_ablation_preset(preset_name)
    train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
    trainer = Trainer(train_ds, test_ds)
    trainer.train()

Applied ablation:
  preset          : baseline_no_lwf
  lwf_enabled     : False
  lambda_lwf      : 0.0
  lambda_lwf_decay: 1.0
  lambda_start    : None
  lambda_end      : None
  n_reset         : 10
  save_model_dir  : ./saved_model_ablations/baseline_no_lwf


In [ ]:
# This cell is kept for manual single-run testing if needed.
# trainer.train()